The contents of this notebook were created with assistance from Claude generative AI.

# Failure Analysis — Candidate Records (Tuned ModernBERT)

Lists misclassified human-gold records for three failure categories, drawn **dynamically** from the
saved tuned-ModernBERT predictions — no records or numbers are hard-coded. For each category the
notebook joins the predictions parquet to the human gold set, keeps the confident misclassifications
(`conf > 0.90`) in that category's `(true -> predicted)` direction, and takes the **5 with the fewest
lines** (then fewest characters). Each candidate is listed with its **row_id, category, confidence,
target/context line counts, posted date and subreddit**; a second cell per category prints the exact
**context + target text the model was given**.

**CPU-only.** Reads `stance_predictions_modernbert_tuned.parquet` (Project Data - 15) and the gold set
from `m5_common.load_frames()`; no model is loaded. The parquet has no labels — the true labels come
from the gold join on `reddit_id == id`.

> Kernel: `mads-m2-classifiers` (any env with pandas + openpyxl works — no GPU needed). The
> `m5_common` import is CPU-safe.

## Setup — load predictions + gold, derive line/confidence columns, define the finder

In [1]:
import sys
P15 = os.getcwd()
P16 = os.getcwd()
if P15 not in sys.path: sys.path.insert(0, P15)
import numpy as np, pandas as pd
import m5_common as M           # CPU-safe import from P15
pd.set_option("display.max_colwidth", 200)

LABELS       = M.LABELS
PRED_PARQUET = "../data/stance_predictions_modernbert_tuned.parquet"
PROBA_COLS   = ["p_anti", "p_neutral", "p_pro", "p_off_topic"]

# gold (true labels) joined to the tuned-ModernBERT predictions on the Reddit id
gold = M.load_frames()["gold_text"].copy()
pred = pd.read_parquet(PRED_PARQUET)
J = gold.merge(pred[["id", "created_utc", "stance_pred", *PROBA_COLS]],
               left_on="reddit_id", right_on="id", how="left")

J["conf"] = J[PROBA_COLS].max(axis=1)                                   # predicted-class confidence

def _nlines(s):
    s = str(s).strip()
    return 0 if not s else s.count("\n") + 1
J["target_lines"]  = J["target"].map(_nlines)
J["context_lines"] = J["context"].map(_nlines)
J["target_chars"]  = J["target"].astype(str).str.strip().str.len()
J["posted"]        = pd.to_datetime(J["created_utc"], unit="s", utc=True).dt.strftime("%Y-%m-%d")  # UTC

# failure categories: (display name, allowed TRUE labels, allowed PREDICTED labels)
CATEGORIES = [
    ("Polarity flip (pro <-> anti)",               {"anti", "pro"}, {"anti", "pro"}),
    ("Off-topic confusion (neutral -> off-topic)", {"neutral"},     {"off-topic"}),
    ("Neutral boundary (neutral -> pro)",          {"neutral"},     {"pro"}),
]
CONF_MIN       = 0.90
N_PER_CATEGORY = 5

def find_candidates(true_set, pred_set, conf_min=CONF_MIN, n=N_PER_CATEGORY):
    '''Confident misclassifications (true in true_set, predicted in pred_set, true != pred, conf >
    conf_min), ranked by fewest target lines then fewest characters; top n. A pure function of the
    parquet + gold join — nothing hand-picked.'''
    c = J[J["label"].isin(true_set) & J["stance_pred"].isin(pred_set)
          & (J["label"] != J["stance_pred"]) & (J["conf"] > conf_min)].copy()
    return c.sort_values(["target_lines", "target_chars"]).head(n)

print(f"gold records       : {len(gold)}")
print(f"matched in parquet : {J['stance_pred'].notna().sum()}")
print(f"selection rule     : conf > {CONF_MIN}, fewest target lines then chars, top {N_PER_CATEGORY} per category")

gold records       : 301
matched in parquet : 301
selection rule     : conf > 0.9, fewest target lines then chars, top 5 per category


## Candidate tables — 5 per category

Each table is produced by `find_candidates(...)` against the live parquet+gold join. Columns:
**row_id · true · pred · confidence · target_lines · context_lines · posted · subreddit**.
Written together to `failure_records.csv`.

In [2]:
DISPLAY = ["row_id", "true", "pred", "confidence",
           "target_lines", "context_lines", "posted", "subreddit"]

frames = []
for name, tset, pset in CATEGORIES:
    c = find_candidates(tset, pset).copy()
    c["category"]   = name
    c["true"]       = c["label"]
    c["pred"]       = c["stance_pred"]
    c["confidence"] = c["conf"].round(3)
    frames.append(c)
    print(f"\n=== {name} ===")
    display(c[DISPLAY].reset_index(drop=True))

candidates = pd.concat(frames, ignore_index=True)
CSV_COLS = ["row_id", "category", "true", "pred", "confidence",
            "target_lines", "context_lines", "posted", "subreddit", "reddit_id", "kind", "target_chars"]
candidates[CSV_COLS].to_csv("failure_records.csv", index=False)
print(f"\nwrote failure_records.csv  ({len(candidates)} candidates: {N_PER_CATEGORY} per category)")

def print_model_inputs(category_name):
    '''Print the exact assembled model-input text ([CONTEXT] ... [TARGET] ...) for every candidate in
    a category — what the tuned ModernBERT was fed (pre-tokenization; left-truncated at 512 tokens,
    not triggered for these short items).'''
    sub = candidates[candidates["category"] == category_name]
    print(f"### {category_name}  —  model-input text for each of the {len(sub)} candidates\n")
    for _, r in sub.iterrows():
        print("=" * 100)
        print(f"{r['row_id']}  |  true={r['true']}  pred={r['pred']}  conf={r['confidence']:.3f}  |  "
              f"r/{r['subreddit']} ({r['kind']})  posted {r['posted']}  "
              f"[target {r['target_lines']} line / context {r['context_lines']} line]")
        print()
        print(str(r["text"]))
        print()


=== Polarity flip (pro <-> anti) ===


,row_id,true,pred,confidence,target_lines,context_lines,posted,subreddit
0,r01695,anti,pro,0.962,1,4,2025-01-06,hudsonvalley
1,r04449,pro,anti,0.988,1,2,2024-06-14,fuckcars
2,r05029,anti,pro,0.953,1,1,2024-06-07,MicromobilityNYC
3,r03288,anti,pro,0.942,1,5,2024-10-27,transit
4,r00134,anti,pro,0.995,1,17,2024-07-31,MicromobilityNYC



=== Off-topic confusion (neutral -> off-topic) ===


,row_id,true,pred,confidence,target_lines,context_lines,posted,subreddit
0,r02680,neutral,off-topic,0.942,1,2,2024-06-16,newyorkcity
1,r01980,neutral,off-topic,0.937,1,1,2025-04-13,nyc
2,r03090,neutral,off-topic,0.997,1,4,2025-01-08,fuckcars
3,r03845,neutral,off-topic,0.906,1,3,2024-09-24,newyorkcity
4,r01781,neutral,off-topic,0.989,1,2,2025-01-02,nycrail



=== Neutral boundary (neutral -> pro) ===


,row_id,true,pred,confidence,target_lines,context_lines,posted,subreddit
0,r00181,neutral,pro,0.998,1,2,2025-02-01,nyc
1,r03464,neutral,pro,0.999,1,1,2024-11-22,jerseycity
2,r04915,neutral,pro,0.985,1,1,2024-06-06,fuckcars
3,r02783,neutral,pro,0.990,1,4,2024-02-27,NYCbike
4,r02209,neutral,pro,0.969,1,2,2024-08-26,MicromobilityNYC



wrote failure_records.csv  (15 candidates: 5 per category)


## Model-input text — Polarity flip (pro ↔ anti)

In [3]:
print_model_inputs(CATEGORIES[0][0])

### Polarity flip (pro <-> anti)  —  model-input text for each of the 5 candidates

r01695  |  true=anti  pred=pro  conf=0.962  |  r/hudsonvalley (comment)  posted 2025-01-06  [target 1 line / context 4 line]

[CONTEXT]
[POST · OP] Hochul wants to triple New York's child tax credit
[A] $1,000 is a "massive increase". Give me a fucking break.

Just build some goddamn housing ffs.

[TARGET]
[B] Would still be less than what I now have to pay thanks to congestion pricing.

r04449  |  true=pro  pred=anti  conf=0.988  |  r/fuckcars (comment)  posted 2024-06-14  [target 1 line / context 2 line]

[CONTEXT]
[POST · OP] The first of these that I've seen in midtown Manhattan
[A] I wouldn't trust speed humps to help much. Never felt the need to slow down for them even in old Corolla. Especially unhelpful with all these damn SUVers and Pickup Truckers. Super low one anyway.

[TARGET]
[OP] I'll take what I can get, the "pause" on congestion pricing has me grasping at straws for relief.

r05029  |  

## Model-input text — Off-topic confusion (neutral → off-topic)

In [4]:
print_model_inputs(CATEGORIES[1][0])

### Off-topic confusion (neutral -> off-topic)  —  model-input text for each of the 5 candidates

r02680  |  true=neutral  pred=off-topic  conf=0.942  |  r/newyorkcity (comment)  posted 2024-06-16  [target 1 line / context 2 line]

[CONTEXT]
[POST · OP] What’s this thing atop the Empire State Building?
    There something greenish wrapping the spire, can’t make out what it is

[TARGET]
[A] Congestion pricing

r01980  |  true=neutral  pred=off-topic  conf=0.937  |  r/nyc (comment)  posted 2025-04-13  [target 1 line / context 1 line]

[CONTEXT]
[POST · OP] New amazon delivery vehicle that requires no license spotted in NYC

[TARGET]
[A] likely the result of congestion pricing.

r03090  |  true=neutral  pred=off-topic  conf=0.997  |  r/fuckcars (comment)  posted 2025-01-08  [target 1 line / context 4 line]

[CONTEXT]
[POST · OP] Bring on the congestion pricing arms race!
    NJ gubernatorial candidate wants NJ to implement their own CP to impact NY drivers coming into NJ. I love it. 

[ht

## Model-input text — Neutral boundary (neutral → pro)

In [5]:
print_model_inputs(CATEGORIES[2][0])

### Neutral boundary (neutral -> pro)  —  model-input text for each of the 5 candidates

r00181  |  true=neutral  pred=pro  conf=0.998  |  r/nyc (comment)  posted 2025-02-01  [target 1 line / context 2 line]

[CONTEXT]
[POST · OP] Has NYC congestion pricing worked? MTA releases dramatic new traffic volume numbers
[A] Looking at mta.info right now, no it hasn't. Signal and door failures galore!

[TARGET]
[B] Congestion pricing is paying for signal improvements doofus

r03464  |  true=neutral  pred=pro  conf=0.999  |  r/jerseycity (comment)  posted 2024-11-22  [target 1 line / context 1 line]

[CONTEXT]
[POST · OP] Fulop is the only candidate ( so far atleast) that is pro transit and anti highway widening

[TARGET]
[A] lol, you can't be pro environment and against congestion pricing.

r04915  |  true=neutral  pred=pro  conf=0.985  |  r/fuckcars (post)  posted 2024-06-06  [target 1 line / context 1 line]

[CONTEXT]
(top-level post — no parent context)

[TARGET]
[OP] Very well made video d